<a href="https://colab.research.google.com/github/Avichay3/Full-training/blob/attention/attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [5]:
class SelfAttentionOneHead(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()

        self.d_model = d_model
        self.d_k = d_k

        # linear layers for q, k, v
        self.q_layer = nn.Linear(d_model, d_k, bias=False)
        self.k_layer = nn.Linear(d_model, d_k, bias=False)
        self.v_layer = nn.Linear(d_model, d_k, bias=False)

    def forward(self, x, mask=None):
        # x shape:(batch_size, seq_len, d_model)

        q = self.q_layer(x)
        k = self.k_layer(x)
        v = self.v_layer(x)

        # attention scores
        scores = torch.matmul(q, k.transpose(1, 2))

        # scale
        scores = scores / math.sqrt(self.d_k)

        # optional mask
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # turn scores into probabilities
        attn_weights = F.softmax(scores, dim=-1)

        # weighted sum of v
        out = torch.matmul(attn_weights, v)

        return out, attn_weights

In [7]:
torch.manual_seed(0)

batch_size = 2
seq_len = 4
d_model = 8
d_k = 8

x = torch.randn(batch_size, seq_len, d_model)

attn = SelfAttentionOneHead(d_model, d_k)
out, weights = attn(x)

print("x shape:", x.shape)
print("out shape:", out.shape)
print("weights shape:", weights.shape)

x shape: torch.Size([2, 4, 8])
out shape: torch.Size([2, 4, 8])
weights shape: torch.Size([2, 4, 4])


In [8]:
print(weights[0])
print()
print(weights[0].sum(dim=-1))

tensor([[0.3631, 0.1601, 0.3278, 0.1490],
        [0.2494, 0.2568, 0.2139, 0.2799],
        [0.2425, 0.2233, 0.2496, 0.2846],
        [0.3087, 0.2223, 0.2323, 0.2367]], grad_fn=<SelectBackward0>)

tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [9]:
# just checking that each row in attention really sums to 1
row_sums = weights[0].sum(dim=-1)
print(row_sums)

tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


# now I make a multi head attention

In [10]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_layer = nn.Linear(d_model, d_model, bias=False)
        self.k_layer = nn.Linear(d_model, d_model, bias=False)
        self.v_layer = nn.Linear(d_model, d_model, bias=False)

        self.out_layer = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        # x shape: (batch_size, seq_len, d_model)
        batch_size = x.shape[0]
        seq_len = x.shape[1]

        q = self.q_layer(x)
        k = self.k_layer(x)
        v = self.v_layer(x)

        # split into heads
        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim)

        # move heads dimension forward
        q = q.transpose(1, 2)  # (batch_size, num_heads, seq_len, head_dim)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1))
        scores = scores / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)

        out = torch.matmul(attn_weights, v)

        # bring back to (batch_size, seq_len, d_model)
        out = out.transpose(1, 2).contiguous()
        out = out.view(batch_size, seq_len, self.d_model)

        out = self.out_layer(out)

        return out, attn_weights

In [12]:
torch.manual_seed(0)

batch_size = 2
seq_len = 5
d_model = 8
num_heads = 2

x = torch.randn(batch_size, seq_len, d_model)

mha = MultiHeadSelfAttention(d_model = d_model, num_heads = num_heads)
out, weights = mha(x)

print("x shape:", x.shape)
print("out shape:", out.shape)
print("weights shape:", weights.shape)

x shape: torch.Size([2, 5, 8])
out shape: torch.Size([2, 5, 8])
weights shape: torch.Size([2, 2, 5, 5])


In [13]:
# just checking one head from one sentence
print(weights[0, 0])

tensor([[0.3057, 0.1223, 0.2659, 0.1278, 0.1782],
        [0.1508, 0.2272, 0.1306, 0.2685, 0.2229],
        [0.1585, 0.2179, 0.1935, 0.2409, 0.1892],
        [0.1958, 0.1870, 0.1491, 0.2279, 0.2402],
        [0.1813, 0.2064, 0.2115, 0.2026, 0.1982]], grad_fn=<SelectBackward0>)


In [14]:
# rows should sum to 1 after softmax
print(weights[0, 0].sum(dim=-1))

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


# Now I'm gonna implement the feed forward part:

In [15]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)

        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)

        return x